In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [2]:
GEN_1_RANDOM = CF_OUTPUTS / "gen1_models_explainers_constraints" / "random_search_exp"
GEN_1_RANDOM.is_dir()

True

In [3]:
batch_data = GEN_1_RANDOM / "XGB_prange_highthres_2026-04-17" / "annotated.csv"
batch_df = pd.read_csv(batch_data)

### set PD options

In [4]:
pd.set_option("display.max_rows", None)

In [5]:
# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [6]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

In [7]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,84.0,,,,0.026,
1,0,cf_1,,,5,,,,15.1784,,,0.2446,2,True,0.026,0.0022
2,0,cf_2,,1,5,,,,,,,0.25,2,True,0.026,0.0045
3,0,cf_3,,,,,,,15.6601,,,0.1044,1,True,0.026,0.004
4,0,cf_4,1,,,,,,18.1644,,,0.1508,2,False,0.026,0.0431
5,0,cf_5,3,,,,,,16.0498,,,0.1339,2,True,0.026,0.0095
6,0,cf_6,,,,7,,,18.7952,,,0.131,2,False,0.026,0.0707
7,0,cf_7,1,2,,,,,,,,0.1875,2,False,0.026,0.0337
8,0,cf_8,,,,6,,,16.1217,,,0.1733,2,False,0.026,0.0347
9,0,cf_9,,,,,,,15.0933,1,,0.2472,2,True,0.026,0.003


In [8]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation
# prefers valid CFs without violations, and lowest Gower
single_gower_df = select_one_cf_per_query(batch_df, selector="gower")
single_gower_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,84.0,,,,0.026,
9,0,cf_3,,,,,,,15.6601,,,0.1044,1,True,0.026,0.004
1,1,xin,3,4,1,2,3,0,22.3757,0,0.19,,,,0.2497,
10,1,cf_4,,,,4,,,,,,0.05,1,True,0.2497,0.0782
2,2,xin,5,3,1,1,4,0,22.694,7,0.23,,,,0.218,
11,2,cf_8,,,,,,,20.7906,,,0.0357,1,True,0.218,0.1046
3,3,xin,3,3,6,6,2,0,24.3418,1,92.97,,,,0.0653,
12,3,cf_9,,,,,,,15.2866,,,0.125,1,True,0.0653,0.0152
4,4,xin,5,4,2,7,1,0,21.2585,3,92.58,,,,0.0811,
13,4,cf_5,,,,,,,21.2361,,,0.0009,1,True,0.0811,0.0141


In [9]:
# Random selection (Gower may pick only BMI variations)
single_random_df = select_one_cf_per_query(batch_df, selector="random")
single_random_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,84.0,,,,0.026,
9,0,cf_1,,,5,,,,15.1784,,,0.2446,2,True,0.026,0.0022
1,1,xin,3,4,1,2,3,0,22.3757,0,0.19,,,,0.2497,
10,1,cf_2,,,,,,,18.1778,,,0.125,1,True,0.2497,0.0486
2,2,xin,5,3,1,1,4,0,22.694,7,0.23,,,,0.218,
11,2,cf_1,,,,,,,17.6974,,,0.0937,1,True,0.218,0.0195
3,3,xin,3,3,6,6,2,0,24.3418,1,92.97,,,,0.0653,
12,3,cf_9,,,,,,,15.2866,,,0.125,1,True,0.0653,0.0152
4,4,xin,5,4,2,7,1,0,21.2585,3,92.58,,,,0.0811,
13,4,cf_9,4,,,,,,18.416,,,0.1439,2,True,0.0811,0.0004
